# NER evaluation and submission

Run the cells in order.

Use the course-provided CoNLL files in `data/conll2003/`; see the README for setup.

## Load the challenge training and validation data

For the challenge experiment we use the repository’s own utilities to read CoNLL files, build a vocabulary from training sentences and create padded batches, keeping a fixed sample of two thousand training sentences and five hundred validation sentences so that the browser can train a small model while validation remains separate from parameter updates.

In [ ]:
import json
import torch
from pathlib import Path
import copy, random, time
from torch import nn
from seqeval.metrics import accuracy_score, precision_score, recall_score, f1_score
from seqeval.metrics import classification_report
from seqeval.metrics.sequence_labeling import get_entities
from ner_project_utils import (
    LABEL_NAMES, FlexibleNERTagger, parse_conll, build_vocab, make_dataloader,
    train_epoch, evaluate_model, save_checkpoint, load_checkpoint,
)


torch.set_num_threads(1)
torch.manual_seed(17)
device = torch.device("cpu")
data_dir = Path("data/conll2003")
training = random.Random(17).sample(parse_conll(data_dir / "train.txt"), 2000)
validation = random.Random(17).sample(parse_conll(data_dir / "valid.txt"), 500)
vocab = build_vocab(training)
train_loader = make_dataloader(training, vocab, batch_size=32, shuffle=True)
valid_loader = make_dataloader(validation, vocab, batch_size=64, shuffle=False)
print("Training sentences:", len(training), "Validation sentences:", len(validation))
print("Vocabulary:", len(vocab), "Label order:", LABEL_NAMES)

**Question:** Why do we map validation words absent from the training vocabulary to UNK instead of adding their own embedding rows before evaluation?

## Build the model the upload service can reconstruct

The challenge loader reconstructs FlexibleNERTagger from the saved embedding width, hidden-layer widths and dropout setting, so we use that class for the submission model even though its embedding, linear layers and ReLU follow the same computation as the handwritten tagger in part one, keeping the configuration consistent with the weights that will be saved.

In [ ]:
config = {
    "embedding_dim": 32,
    "hidden_dims": [64],
    "dropout": 0.1,
    "learning_rate": 0.01,
    "epochs": 3,
    "train_subset_size": len(training),
}
model = FlexibleNERTagger(
    vocab_size=len(vocab), num_labels=len(LABEL_NAMES),
    embedding_dim=config["embedding_dim"], hidden_dims=config["hidden_dims"],
    dropout=config["dropout"],
).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

**Question:** If we change hidden_dims after training but keep the old weight tensors, why would the evaluator fail when it reconstructs the model?

## Train and save the best validation checkpoint

After each epoch we evaluate the model on the held-out validation sample and save its weights whenever entity F1 improves, which chooses a checkpoint using performance on examples excluded from parameter updates rather than the lowest training loss, while the test split remains outside the training and model-selection loop.

In [ ]:
best_f1, best_state = -1.0, None
history = []
started = time.perf_counter()
for epoch in range(config["epochs"]):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    result = evaluate_model(model, valid_loader, criterion, device, LABEL_NAMES)
    history.append({"epoch": epoch + 1, "train_loss": float(train_loss),
                    "validation_f1": float(result.entity_f1)})
    print(f"Epoch {epoch+1}: loss={train_loss:.4f}, validation F1={result.entity_f1:.1%}")
    if result.entity_f1 > best_f1:
        best_f1 = float(result.entity_f1)
        best_state = copy.deepcopy(model.state_dict())
        save_checkpoint("best_student_model.pt", model, vocab, config,
                        {"validation_entity_f1": best_f1})
model.load_state_dict(best_state)
print("Selected validation F1:", best_f1)
print("Training seconds:", round(time.perf_counter() - started, 2))

**Question:** If the final epoch has lower validation F1 than an earlier one, which epoch should best_student_model.pt contain and why?

## Check what the checkpoint contains

A checkpoint needs the learned weights together with the vocabulary, label order and architecture configuration because the evaluator must reproduce both the network and the mapping between words, IDs and predicted tags, so we reload the saved file through the official loader and compare its scores with the selected model before downloading it.

In [ ]:
checkpoint = torch.load("best_student_model.pt", map_location="cpu", weights_only=True)
print("Saved fields:", sorted(checkpoint))
restored, restored_vocab, metadata = load_checkpoint("best_student_model.pt", device)
probe = next(iter(valid_loader))["input_ids"]
model.eval()
restored.eval()
with torch.no_grad():
    identical = torch.equal(model(probe), restored(probe))
assert identical and restored_vocab == vocab
assert metadata["label_names"] == LABEL_NAMES
size = Path("best_student_model.pt").stat().st_size
print("Reloaded scores identical:", identical)
print("File:", Path("best_student_model.pt").name, "Bytes:", size)

**Question:** If we save only model.state_dict(), which information would the evaluator still need to turn a new sentence into predicted tag names?

### Alternative A

If the weight tensors and output shape stay unchanged but the saved label names are reversed, will the predicted names still describe the same classes, and would a tensor-shape check detect the problem?

In [ ]:
with torch.no_grad():
    alt_ids = restored(probe).argmax(dim=-1)[0].tolist()
alt_names = list(reversed(LABEL_NAMES))
alt_correct_tags = [LABEL_NAMES[index] for index in alt_ids]
alt_wrong_tags = [alt_names[index] for index in alt_ids]
print("Same output width:", len(alt_names) == len(LABEL_NAMES))
print("Same decoded tags:", alt_correct_tags == alt_wrong_tags)
print("Working labels:", alt_correct_tags[:12])
print("Alternative labels:", alt_wrong_tags[:12])

## Compare gold and predicted tags

After training, we evaluate the selected model on the real CoNLL validation sample and choose a sentence containing a multiword entity and prediction errors, so that the gold tags, predicted tags and metric calculations all describe an actual model result that students can inspect rather than an invented sentence or a hand-edited prediction.

In [ ]:
validation_results = evaluate_model(model, valid_loader, criterion, device, LABEL_NAMES)
# Prefer a complete sentence with correct, extra and missed entity spans.
candidates = []
for index, example in enumerate(validation):
    tags = example["ner_tags"]
    guesses = validation_results.entity_pred_labels[index]
    expected = set(get_entities(tags))
    found = set(get_entities(guesses))
    if 6 <= len(tags) <= 24 and example["tokens"][-1] == ".":
        matched, extra, missed = len(expected & found), len(found - expected), len(expected - found)
        priority = (matched == 1 and extra == 1 and missed == 2,
                    matched > 0 and extra > 0 and missed > 0,
                    any(end > start for _, start, end in expected), -len(tags))
        candidates.append((priority, -index, index))
example_index = max(candidates)[2] if candidates else 0
example = validation[example_index]
words = example["tokens"]
gold = [example["ner_tags"]]
predicted = [validation_results.entity_pred_labels[example_index]]
print("CoNLL validation sample row:", example_index)
for row in zip(words, gold[0], predicted[0]):
    print(row)

**Question:** Looking at the actual word-and-tag rows, which complete entities did the model recover, which did it miss, and which predicted entity ends before the gold span ends?

## Token accuracy

Token accuracy counts how many real word positions receive the correct tag and divides by the number of evaluated words, so we first calculate it for the selected validation sentence and then compare an all-O prediction against every sentence in the five-hundred-sentence validation sample to measure how many correct tags can be obtained simply by predicting the most common class.

In [ ]:
correct = sum(a == b for a, b in zip(gold[0], predicted[0]))
total = len(gold[0])
print(f"Selected sentence accuracy: {correct}/{total} = {correct / total:.1%}")

validation_gold = validation_results.entity_gold_labels
validation_predicted = validation_results.entity_pred_labels
all_o = [["O"] * len(tags) for tags in validation_gold]
all_o_correct = sum(tag == "O" for tags in validation_gold for tag in tags)
all_o_total = sum(map(len, validation_gold))
print("Validation tokens:", all_o_total, "Outside-entity tokens:", all_o_correct)
print("All-O token accuracy:", accuracy_score(validation_gold, all_o))
print("All-O entity F1:", f1_score(validation_gold, all_o, zero_division=0))

**Question:** When the all-O predictions are scored against this validation sample, how does the proportion of O tags explain the accuracy, and why does the entity score remain zero?

### Alternative A

If we count accuracy only where the gold tag belongs to an entity, does the resulting fraction measure the same token accuracy as the challenge, or has the denominator changed?

In [ ]:
alt_pairs = [(actual, guess) for actual, guess in zip(gold[0], predicted[0])
             if actual != "O"]
alt_correct = sum(actual == guess for actual, guess in alt_pairs)
print("Alternative accuracy:", alt_correct / len(alt_pairs))
print("Positions counted:", len(alt_pairs))
print("Challenge token accuracy:", accuracy_score(gold, predicted))

## Count exact entity matches

Entity scoring compares the type and boundaries of each real predicted span with the corresponding gold spans from the selected sentence, counting an exact match as a true positive, an unmatched prediction as a false positive and an unmatched gold span as a false negative, which lets us identify precisely how the observed word-level mistakes affect complete entities.

In [ ]:
def entity_set(sentences):
    return {(sentence, kind, start, end)
            for sentence, tags in enumerate(sentences)
            for kind, start, end in get_entities(tags)}

gold_entities = entity_set(gold)
predicted_entities = entity_set(predicted)
tp = len(gold_entities & predicted_entities)
fp = len(predicted_entities - gold_entities)
fn = len(gold_entities - predicted_entities)
print("Gold entities:", sorted(gold_entities))
print("Predicted entities:", sorted(predicted_entities))
print("TP:", tp, "FP:", fp, "FN:", fn)

**Question:** For a real prediction whose entity ends before the gold span ends, why does the span comparison count both an extra prediction and a missed gold entity?

### Alternative A

If we count a prediction as correct whenever it overlaps a gold span of the same type, would the shortened name receive credit, and would that still match the challenge’s exact entity scoring?

In [ ]:
alt_matches = sum(
    any(ps == gs and pk == gk and max(pa, ga) <= min(pb, gb)
        for gs, gk, ga, gb in gold_entities)
    for ps, pk, pa, pb in predicted_entities
)
print("Overlap matches:", alt_matches)
print("Exact matches:", tp)
print("Predicted entities:", len(predicted_entities))

## Precision, recall and F1

Precision measures how many predicted entities are correct by dividing true positives by all predicted entities, while recall measures how many gold entities were recovered by dividing true positives by all gold entities, and F1 combines those two fractions using their harmonic mean so that a low value for either one reduces the combined score.

In [ ]:
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print(f"Precision = TP/(TP+FP) = {tp}/{tp+fp} = {precision:.3f}")
print(f"Recall    = TP/(TP+FN) = {tp}/{tp+fn} = {recall:.3f}")
print(f"F1 = 2*P*R/(P+R) = {f1:.3f}")
print("Equivalent count formula:", 2 * tp / (2 * tp + fp + fn))

**Question:** Using the counts from this validation sentence, how do precision and recall answer different questions about the entities that were predicted and the entities that were present?

### Alternative A

If we combine precision and recall using their ordinary arithmetic mean, will that produce the F1 score defined by seqeval, and how can the true-positive, false-positive and false-negative counts check the result?

In [ ]:
alt_average = (precision + recall) / 2
alt_count_f1 = 2 * tp / (2 * tp + fp + fn)
print("Arithmetic average:", round(alt_average, 6))
print("F1 from entity counts:", round(alt_count_f1, 6))
print("Seqeval F1:", round(f1_score(gold, predicted, zero_division=0), 6))

## Use seqeval to calculate the metrics

seqeval is a Python evaluation library that extracts entities from lists of gold and predicted tags and calculates their exact-match precision, recall and F1, so we first verify its answer against our hand calculation for the selected validation sentence and then inspect the per-type report across the full validation sample to see whether the same errors occur elsewhere.

In [ ]:
measured = {
    "token_accuracy": accuracy_score(gold, predicted),
    "entity_precision": precision_score(gold, predicted, zero_division=0),
    "entity_recall": recall_score(gold, predicted, zero_division=0),
    "entity_f1": f1_score(gold, predicted, zero_division=0),
}
print(measured)
print("Selected sentence report:")
print(classification_report(gold, predicted, digits=3, zero_division=0))
print("Full 500-sentence validation report:")
print(classification_report(validation_gold, validation_predicted, digits=3, zero_division=0))
assert abs(measured["entity_precision"] - precision) < 1e-9
assert abs(measured["entity_recall"] - recall) < 1e-9
assert abs(measured["entity_f1"] - f1) < 1e-9

**Question:** If we join several sentences into a single tag list, what information would seqeval lose about where entities are allowed to continue?

### Alternative A

If we swap the gold and predicted arguments when calling seqeval, will precision and recall keep the same meaning, and could an unchanged F1 conceal the mistake?

In [ ]:
alt_precision = precision_score(predicted, gold, zero_division=0)
alt_recall = recall_score(predicted, gold, zero_division=0)
alt_f1 = f1_score(predicted, gold, zero_division=0)
print("Reversed precision:", round(alt_precision, 6))
print("Reversed recall:", round(alt_recall, 6))
print("Reversed F1:", round(alt_f1, 6))
print("Working F1:", round(f1_score(gold, predicted, zero_division=0), 6))

## Use the challenge scoring convention

The official challenge calculates seqeval scores in its default conlleval-compatible mode, while strict IOB2 scoring also checks whether predicted B and I tags form valid entity starts and continuations, so we apply both modes to the trained model’s real validation predictions and count the malformed I starts that can make the two scores differ.

In [ ]:
from seqeval.scheme import IOB2
invalid_starts = 0
for sentence in validation_predicted:
    previous = "O"
    for tag in sentence:
        if tag.startswith("I-") and previous not in ("B-" + tag[2:], "I-" + tag[2:]):
            invalid_starts += 1
        previous = tag
default_f1 = f1_score(validation_gold, validation_predicted, zero_division=0)
strict_f1 = f1_score(validation_gold, validation_predicted, mode="strict", scheme=IOB2, zero_division=0)
print("Malformed I starts in actual predictions:", invalid_starts)
print("Challenge/default validation F1:", default_f1)
print("Strict IOB2 validation F1:", strict_f1)

**Question:** Why must our validation selection use the same scoring convention as the evaluator that will rank the submitted checkpoint?

## Run the official evaluator before uploading

Once the training choices are fixed, we run the repository’s unchanged evaluator on the saved checkpoint and the full test split as required by the submission page, which checks loading, predictions and scoring through the same evaluation code used by the challenge rather than relying on the five-hundred-sentence validation score recorded during training.

In [ ]:
import runpy, sys
previous_arguments = sys.argv[:]
try:
    sys.argv = ["evaluate_ner_model.py", "--checkpoint", "best_student_model.pt",
                "--split", "test", "--device", "cpu",
                "--output-json", "evaluation_results.json",
                "--save-confusion", "confusion_matrix.npy"]
    runpy.run_path("evaluate_ner_model.py", run_name="__main__")
finally:
    sys.argv = previous_arguments

**Question:** Why can this final test F1 differ from the validation F1 that selected the checkpoint even though both evaluations load the same weights?

For the same verification in a fresh terminal process, run:

```sh
python evaluate_ner_model.py --checkpoint best_student_model.pt --split test --device cpu --output-json evaluation_results.json --save-confusion confusion_matrix.npy
```

Use `--split validation` while comparing settings, then perform the required test verification after selecting the model.

## Read the evaluator output

The evaluator writes a JSON summary containing token accuracy, entity metrics, parameter count, elapsed verification time and peak memory, along with a token confusion matrix whose rows represent true tags and columns represent predicted tags, so we inspect both the entity score used for ranking and the kinds of token mistakes that score alone cannot describe.

In [ ]:
import numpy as np
summary = json.loads(Path("evaluation_results.json").read_text())
metrics = summary["metrics"]
for name in ["token_accuracy", "entity_precision", "entity_recall", "entity_f1"]:
    print(name, f"{metrics[name]:.1%}")
print("Split:", summary["split"], "Device:", summary["device"])
print("Parameters:", summary["num_parameters"])
print("Verification seconds:", round(summary["verification_time_seconds"], 3))
print("Peak memory MB:", summary["peak_memory_mb"])
confusion = np.load("confusion_matrix.npy")
print("Confusion shape:", confusion.shape)

**Question:** If a true B-ORG row assigns a large fraction to the O column, what does that tell us about the model’s organisation predictions?

## Submit the checkpoint

1. After running the official evaluator successfully, download `best_student_model.pt` using the button below the website lesson, or find that same filename in the folder where you ran the local notebook.
2. Open [the NER challenge](https://neuralisa.com/ner-challenge/), enter the email address supplied for your course and press **Continue**, then choose a password on first use or enter your existing password so that the page can show your assigned alias.
3. Under **Submit your model**, choose the file named `best_student_model.pt`, tick **I tested this checkpoint with the official evaluation script**, and press **Upload checkpoint** to send that file for server evaluation.
4. Wait while the page reports the queued or evaluating state, then inspect the resulting entity F1, precision, recall, token accuracy, evaluation time and memory, using **Your submissions** to find the record and the class leaderboard to compare your alias’s best score.

The upload page accepts a nonempty checkpoint up to 60 MiB, and its evaluator rebuilds the model through `FlexibleNERTagger` using the saved configuration, vocabulary and label order; the part-one BiLSTM demonstration therefore needs an evaluator update before its weights can be submitted through this interface.


Upload `best_student_model.pt` at [the NER challenge](https://neuralisa.com/ner-challenge/).